# Provider-Validation Resolution Floor (delta) and Section 5 Results

Independent Python reproduction of `experiments/exp4/analyze-provider-validation.js`
(the original Node.js analysis run against live Firestore data). This notebook needs no
database credentials or live access -- it downloads its own frozen, public CSV from the
Beyond-The-Mean GitHub repo's `data/` folder if not found locally (same pattern as
`Exp4_HRS_IntegratedW_Association_Git.ipynb`), verifies it against a locked SHA-256 hash,
and reruns the full analysis from there. Anyone can rerun this top-to-bottom to confirm the
delta and Section 5 numbers reported in `PROVIDER_VALIDATION_PRESPEC.md` Section 9 and in
`resolution_floor_derivation-provider-validation.md`.

Design this data comes from: `PROVIDER_VALIDATION_PRESPEC.md`. 269 completed machine-only
sittings (no human or AI involvement), 30 blocks each, QRNG provider identity
(Outshift vs LFDR) explicitly controlled and logged per block, balanced 15/15 within each
sitting.

`hurst_approx` below is a direct Python port of the JS implementation in
`experiments/exp4/check-hrs-common-mode-cancellation.js` (itself verbatim from
`experiments/exp4/src/stats/coherence.js`) -- same arithmetic, not a reimplementation from
a different starting point.

In [1]:
import os
import hashlib
import urllib.request
import numpy as np
import pandas as pd

CSV_FILENAME = "Frozen_ProviderValidation_Blocks_2026-09-06.csv"
DOWNLOAD_BASE = "https://raw.githubusercontent.com/catboxer/Beyond-The-Mean/main/data"
EXPECTED_SHA256 = "4923bae8217e8fc0c8f266910aca7ee0a860cf856f5a5ce52c4235cc745efcf5"

def sha256_file(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def locate_or_download(filename, download_base=DOWNLOAD_BASE, search_root=None):
    """Look for filename in the working directory, its data/ subfolder, or a sibling
    ../data/ folder (this repo's actual layout: notebook in analysis/, data in ../data/);
    download from the Beyond-The-Mean repo's data/ folder on GitHub if not found locally."""
    root = os.getcwd() if search_root is None else search_root
    candidates = (
        os.path.join(root, filename),
        os.path.join(root, "data", filename),
        os.path.join(root, "..", "data", filename),
    )
    for candidate in candidates:
        if os.path.isfile(candidate):
            return candidate
    dest_dir = os.path.join(root, "data")
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, filename)
    url = f"{download_base}/{filename}"
    print(f"{filename}: not found locally (checked {len(candidates)} candidate paths) -- downloading {url}")
    urllib.request.urlretrieve(url, dest)
    return dest

CSV_PATH = locate_or_download(CSV_FILENAME)
actual_hash = sha256_file(CSV_PATH)
print(f"Loaded from: {CSV_PATH}")
print(f"SHA-256: {actual_hash}")
if actual_hash != EXPECTED_SHA256:
    raise ValueError(f"Hash mismatch on {CSV_FILENAME}: expected {EXPECTED_SHA256}, got {actual_hash}")
print("Hash verified against provider_validation_manifest.json.")

def hurst_approx(bits):
    n = len(bits)
    if n < 20:
        return 0.5
    x = np.where(np.asarray(bits) != 0, 1.0, -1.0)
    mean = x.mean()
    y = minY = maxY = s2 = 0.0
    for d in (x - mean):
        y += d
        if y < minY: minY = y
        if y > maxY: maxY = y
        s2 += d * d
    R = maxY - minY
    S = np.sqrt(s2 / n) if s2 > 0 else 1.0
    if not np.isfinite(S) or S == 0: S = 1.0
    ratio = (R / S) if (R / S) != 0 else 1.0
    if not np.isfinite(ratio) or ratio == 0: ratio = 1.0
    H = np.log(ratio) / np.log(n)
    if not np.isfinite(H): H = 0.5
    return float(max(0.0, min(1.0, H)))


Loaded from: /Users/macuser/Desktop/WTQ/BTM-ZEN/Beyond-The-Mean/analysis/../data/Frozen_ProviderValidation_Blocks_2026-09-06.csv
SHA-256: 4923bae8217e8fc0c8f266910aca7ee0a860cf856f5a5ce52c4235cc745efcf5
Hash verified against provider_validation_manifest.json.


## Load data and reconstruct per-block quantities

In [2]:
df = pd.read_csv(CSV_PATH, dtype={"bits": str})
df["sitting_idx"] = df["sitting_id"].astype("category").cat.codes
n_sit = df["sitting_idx"].nunique()
print(f"Loaded {len(df)} blocks, {n_sit} sittings")

bits_arr = df["bits"].apply(lambda s: np.array([int(c) for c in s], dtype=np.int8))

def half(bits, subject_first, which):
    h1, h2 = bits[1:151], bits[151:301]
    if which == "subject":
        return h1 if subject_first else h2
    return h2 if subject_first else h1

subject_bits = [half(b, sf, "subject") for b, sf in zip(bits_arr, df["subject_gets_first_half"])]
demon_bits = [half(b, sf, "demon") for b, sf in zip(bits_arr, df["subject_gets_first_half"])]
full_bits = [b[1:] for b in bits_arr]  # 300 raw bits, assignment bit excluded

df["hurst_subject"] = [hurst_approx(b) for b in subject_bits]
df["hurst_demon"] = [hurst_approx(b) for b in demon_bits]
df["hurst_combined"] = [hurst_approx(b) for b in full_bits]

df["clean_delta_hit"] = df["hits"] / df["n"] - df["demon_hits"] / df["n"]
df["clean_delta_hurst"] = df["hurst_subject"] - df["hurst_demon"]
df["combined_hit_rate"] = (df["hits"] + df["demon_hits"]) / (2 * df["n"])


Loaded 8042 blocks, 269 sittings


## Part 1 — Resolution floor (delta)

Whole-sitting bootstrap (3,000 draws) on the real, unperturbed subject/PCS paired
difference -- same method as `experiments/exp4/resolution_floor_derivation-exp4.md`
(the companion injection-study derivation), applied here to this study's own data.

In [3]:
N_BOOT = 3000
rng = np.random.default_rng(20260906)

def whole_sitting_bootstrap_pooled(values_by_sitting, n_sit, n_boot, rng):
    obs = np.concatenate(values_by_sitting).mean()
    draws = np.empty(n_boot)
    for d in range(n_boot):
        picks = rng.integers(0, n_sit, size=n_sit)
        draws[d] = np.concatenate([values_by_sitting[i] for i in picks]).mean()
    lo, hi = np.percentile(draws, [2.5, 97.5])
    return obs, lo, hi, (hi - lo) / 2

grouped_hit = [np.array(v) for v in df.groupby("sitting_idx")["clean_delta_hit"].apply(list)]
grouped_hurst = [np.array(v) for v in df.groupby("sitting_idx")["clean_delta_hurst"].apply(list)]

obs_hit, lo_hit, hi_hit, hw_hit = whole_sitting_bootstrap_pooled(grouped_hit, n_sit, N_BOOT, rng)
obs_hurst, lo_hurst, hi_hurst, hw_hurst = whole_sitting_bootstrap_pooled(grouped_hurst, n_sit, N_BOOT, rng)

print(f"clean_delta_hit:   mean={obs_hit:.6f}  95% CI=[{lo_hit:.6f}, {hi_hit:.6f}]  half-width={hw_hit:.6f}")
print(f"clean_delta_hurst: mean={obs_hurst:.6f}  95% CI=[{lo_hurst:.6f}, {hi_hurst:.6f}]  half-width={hw_hurst:.6f}")

delta = max(hw_hit, hw_hurst)
print(f"\ndelta (conservative, larger of the two, rounded up in the writeup to 0.0014) = {delta:.6f}")


clean_delta_hit:   mean=-0.000830  95% CI=[-0.002056, 0.000378]  half-width=0.001217
clean_delta_hurst: mean=0.000199  95% CI=[-0.001212, 0.001579]  half-width=0.001396

delta (conservative, larger of the two, rounded up in the writeup to 0.0014) = 0.001396


## Part 2 — Section 5 Step 1: unpaired Outshift vs LFDR

Combined per-block hit rate = (hits + demon_hits) / 300; combined H_RS = `hurst_approx`
on the full 300-bit raw call (assignment bit excluded). Grouped by `provider_actual`,
whole-sitting bootstrap CI on the difference.

In [4]:
def unpaired_bootstrap(df, valuecol, n_sit, n_boot, rng):
    out_map = {i: np.array(v) for i, v in df[df.provider_actual == "outshift"].groupby("sitting_idx")[valuecol].apply(list).items()}
    lfdr_map = {i: np.array(v) for i, v in df[df.provider_actual == "lfdr"].groupby("sitting_idx")[valuecol].apply(list).items()}
    out_vals = df[df.provider_actual == "outshift"][valuecol].values
    lfdr_vals = df[df.provider_actual == "lfdr"][valuecol].values
    obs_diff = out_vals.mean() - lfdr_vals.mean()
    draws = np.empty(n_boot)
    for d in range(n_boot):
        picks = rng.integers(0, n_sit, size=n_sit)
        o = np.concatenate([out_map.get(i, np.array([])) for i in picks])
        l = np.concatenate([lfdr_map.get(i, np.array([])) for i in picks])
        draws[d] = o.mean() - l.mean()
    lo, hi = np.percentile(draws, [2.5, 97.5])
    return obs_diff, lo, hi, (hi - lo) / 2, len(out_vals), len(lfdr_vals), out_vals.mean(), lfdr_vals.mean()

res_hit1 = unpaired_bootstrap(df, "combined_hit_rate", n_sit, N_BOOT, rng)
res_hurst1 = unpaired_bootstrap(df, "hurst_combined", n_sit, N_BOOT, rng)

for label, res, d in [("hit rate", res_hit1, hw_hit), ("H_RS", res_hurst1, hw_hurst)]:
    obs, lo, hi, hw, nO, nL, mO, mL = res
    excl0 = lo > 0 or hi < 0
    within = lo >= -d and hi <= d
    print(f"{label}: Outshift n={nO} mean={mO:.6f}  LFDR n={nL} mean={mL:.6f}")
    print(f"  diff={obs:.6f}  95% CI=[{lo:.6f}, {hi:.6f}]  excludes0={excl0}  within(+/-{d:.5f} this study's own delta)={within}")


hit rate: Outshift n=4021 mean=0.499551  LFDR n=4021 mean=0.499820
  diff=-0.000269  95% CI=[-0.001469, 0.000961]  excludes0=False  within(+/-0.00122 this study's own delta)=False
H_RS: Outshift n=4021 mean=0.527416  LFDR n=4021 mean=0.528334
  diff=-0.000919  95% CI=[-0.002644, 0.000935]  excludes0=False  within(+/-0.00140 this study's own delta)=False


## Part 3 — Section 5 Step 2: within-sitting paired delta, provider as pairing factor

Per sitting (each has exactly 15 Outshift + 15 LFDR blocks by design): paired delta =
mean(Outshift blocks) - mean(LFDR blocks). Bootstrap over the 269 per-sitting deltas.

In [5]:
def paired_by_provider(df, valuecol, n_sit, n_boot, rng):
    per_sitting = []
    for i, g in df.groupby("sitting_idx"):
        o = g[g.provider_actual == "outshift"][valuecol]
        l = g[g.provider_actual == "lfdr"][valuecol]
        if len(o) == 0 or len(l) == 0:
            continue
        per_sitting.append(o.mean() - l.mean())
    per_sitting = np.array(per_sitting)
    obs = per_sitting.mean()
    n = len(per_sitting)
    draws = np.empty(n_boot)
    for d in range(n_boot):
        draws[d] = per_sitting[rng.integers(0, n, size=n)].mean()
    lo, hi = np.percentile(draws, [2.5, 97.5])
    return obs, lo, hi, (hi - lo) / 2, n

res_hit2 = paired_by_provider(df, "combined_hit_rate", n_sit, N_BOOT, rng)
res_hurst2 = paired_by_provider(df, "hurst_combined", n_sit, N_BOOT, rng)

for label, res, d in [("hit rate", res_hit2, hw_hit), ("H_RS", res_hurst2, hw_hurst)]:
    obs, lo, hi, hw, n = res
    excl0 = lo > 0 or hi < 0
    within = lo >= -d and hi <= d
    print(f"{label}: n_sittings={n}  observed mean paired delta={obs:.6f}")
    print(f"  95% CI=[{lo:.6f}, {hi:.6f}]  excludes0={excl0}  within(+/-{d:.5f} this study's own delta)={within}")


hit rate: n_sittings=269  observed mean paired delta=-0.000430
  95% CI=[-0.001730, 0.000820]  excludes0=False  within(+/-0.00122 this study's own delta)=False
H_RS: n_sittings=269  observed mean paired delta=-0.001125
  95% CI=[-0.003013, 0.000737]  excludes0=False  within(+/-0.00140 this study's own delta)=False


## Reading

Every point estimate here matches the original Node.js run
(`analyze-provider-validation.js`) exactly; bootstrap CI half-widths match to within Monte
Carlo noise (independent RNG streams, same method and data). See
`resolution_floor_derivation-provider-validation.md` for the full writeup: delta ≈ 0.0014,
and both Section 5 steps land as **indeterminate** (no detected provider effect, but not a
clean equivalence pass at this design's own resolution floor).